In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
import os
import random

#from masking_utils import apply_center_mask

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG, HOLE, EPOCHS, BS, LR = 128, 64, 100, 64, 2e-4

In [ ]:
class FlatCEDataset(Dataset):
    def __init__(self, folder_path):
        self.folder_path = folder_path
        self.tfm = T.Compose([
            T.Resize((IMG, IMG)), 
            T.ToTensor(), 
            T.Normalize([.5]*3, [.5]*3)
        ])
        
        valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
        try:
            self.files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_exts)]
        except FileNotFoundError:
            print(f"Directory '{folder_path}' not found. Generating dummy data.")
            self.files = []
            self.n = 2000

    def __len__(self): 
        return len(self.files) if self.files else self.n

    def __getitem__(self, i):
        if self.files:
            img_path = os.path.join(self.folder_path, self.files[i])
            img = Image.open(img_path).convert('RGB')
            img = self.tfm(img)
        else:
            img = torch.rand(3, IMG, IMG) * 2 - 1
            
        # 1. Calculate random coordinates
        max_idx = IMG - HOLE
        start_y = random.randint(0, max_idx)
        start_x = random.randint(0, max_idx)
        
        # 2. Apply the mask and grab the ground truth patch
        masked = img.clone()
        masked[:, start_y:start_y+HOLE, start_x:start_x+HOLE] = 0
        gt = img[:, start_y:start_y+HOLE, start_x:start_x+HOLE]
        
        # 3. Return the coordinates too!
        return masked, gt, img, start_y, start_x

'''
# Point this directly to your folder of images!
INPUT_FOLDER = './natural_images/airplane' 
loader = DataLoader(FlatCEDataset(INPUT_FOLDER), BS, shuffle=True, drop_last=True)
print(f"Dataset loaded with {len(loader.dataset)} images.")
'''

# Setup your two folders
TRAIN_FOLDER = './plant_dataset_all'
TEST_FOLDER = './plant_dataset_final_test' 

# Create two separate loaders
train_loader = DataLoader(FlatCEDataset(TRAIN_FOLDER), BS, shuffle=True, drop_last=True)

# For the test loader, batch size is 10 (your 10 test images), and drop_last is False!
test_loader = DataLoader(FlatCEDataset(TEST_FOLDER), batch_size=12, shuffle=False, drop_last=False)

print(f"Training on {len(train_loader.dataset)} images.")
print(f"Testing on {len(test_loader.dataset)} images.")

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        def blk(ci,co,s=2): return [nn.Conv2d(ci,co,4,s,1), nn.BatchNorm2d(co), nn.LeakyReLU(.2,True)]
        self.net = nn.Sequential(*blk(3,64),*blk(64,128),*blk(128,256),*blk(256,512),*blk(512,512))
        self.fc  = nn.Linear(512*4*4, 4000)
    def forward(self,x): return self.fc(self.net(x).flatten(1))

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(4000,512*4*4), nn.BatchNorm1d(512*4*4), nn.ReLU(True))
        def ubl(ci,co): return [nn.ConvTranspose2d(ci,co,4,2,1), nn.BatchNorm2d(co), nn.ReLU(True)]
        self.net = nn.Sequential(*ubl(512,256),*ubl(256,128),*ubl(128,64),*ubl(64,32),
                                  nn.ConvTranspose2d(32,3,3,1,1), nn.Tanh())
    def forward(self,z): return self.net(self.proj(z).view(-1,512,4,4))

class Disc(nn.Module):
    def __init__(self):
        super().__init__()
        def blk(ci,co,bn=True): return [nn.Conv2d(ci,co,4,2,1),*([ nn.BatchNorm2d(co)] if bn else []),nn.LeakyReLU(.2,True)]
        self.net = nn.Sequential(*blk(3,64,False),*blk(64,128),*blk(128,256),*blk(256,512),
                                  nn.Conv2d(512,1,4,1,0), nn.Sigmoid())
    def forward(self,x): return self.net(x).view(x.size(0))

E, D, Dsc = Encoder().to(device), Decoder().to(device), Disc().to(device)
optG = optim.Adam(list(E.parameters())+list(D.parameters()), LR,   betas=(.5,.999))
optD = optim.Adam(Dsc.parameters(),                           LR/10, betas=(.5,.999))
bce, mse = nn.BCELoss(), nn.MSELoss()

print(f'Params  Enc+Dec: {sum(p.numel() for p in list(E.parameters())+list(D.parameters())):,}  Disc: {sum(p.numel() for p in Dsc.parameters()):,}')

In [5]:
history = []
print("Starting Training...")
for ep in range(1, EPOCHS+1):
    E.train(); D.train(); Dsc.train()
    gl = dl = 0
    for masked, gt, full, start_y, start_x in train_loader:
        masked, gt, full = masked.to(device), gt.to(device), full.to(device)
        fake = D(E(masked))
        
        # Discriminator Step
        optD.zero_grad()
        (bce(Dsc(gt),   torch.ones(BS,device=device)) +
         bce(Dsc(fake.detach()), torch.zeros(BS,device=device))).mul(.5).backward()
        optD.step()
        
        # Generator Step
        optG.zero_grad()
        loss = .999*mse(fake,gt) + .001*bce(Dsc(fake), torch.ones(BS,device=device))
        loss.backward(); optG.step()
        
        gl += loss.item(); dl += 1
        
    avg = gl/dl; history.append(avg)
    if ep % 10 == 0: print(f'Epoch {ep:3d} | Average Loss: {avg:.4f}')

print('Training Complete!')

KeyboardInterrupt: 

In [ ]:
E.eval(); D.eval()

# Catch the random coordinates from your test loader
masked, gt, full, start_y, start_x = next(iter(test_loader))

# Changed to process 12 images
with torch.no_grad(): pred = D(E(masked[:12].to(device))).cpu()

composed = masked[:12].clone()

# Loop through the 12 test images and paste the patch into its unique random spot
for i in range(12):
    y = start_y[i].item() # .item() converts the PyTorch tensor back to a standard integer
    x = start_x[i].item()
    composed[i, :, y:y+HOLE, x:x+HOLE] = pred[i]
    
dn = lambda t: (t*.5+.5).clamp(0,1).permute(1,2,0).numpy()

# Changed the subplot grid to 3 rows by 12 columns, and made the figure wider
fig, axes = plt.subplots(3, 12, figsize=(24, 6))
for i in range(12):
    for j,(img,ttl) in enumerate(zip([masked[i],composed[i],full[i]],['Input','Inpainted','GT'])):
        axes[j,i].imshow(dn(img)); axes[j,i].axis('off')
        if i==0: axes[j,i].set_title(ttl,fontweight='bold')
plt.tight_layout()
plt.show()

plt.figure()
plt.plot(history)
plt.title('Generator Loss Over Time')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(alpha=.3)
plt.show()